# 1. TOA5 data to spectra, corrections, and output plots

Run every cell in order. This example uses the bundled **synthetic** 30-minute,
20 Hz IRGASON file (36,000 samples, with injected spikes); no download is needed.
It demonstrates processing, not validation of a field deployment.

From the repository root install `pip install -e ".[notebooks]"`, then launch
`jupyter notebook`. Select the Python environment where you installed the package.
Plots appear inline and are saved as PNG/PDF under `examples/outputs/toa5/`.
Rerunning replaces files in that output folder.

Workflow: read → inspect → despike → process → diagnose → plot → correct → export.

In [1]:
OUTPUT_NAME = "toa5"
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display

# Works when Jupyter starts in the repository root or examples/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "TaylorSwift").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the TaylorSwift checkout.")
sys.path.insert(0, str(ROOT / "src"))
import TaylorSwift as tswift

%matplotlib inline
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})
OUTPUT = ROOT / "examples" / "outputs" / OUTPUT_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"TaylorSwift {tswift.__version__}; outputs: {OUTPUT}")

def save_plot(fig, name):
    # Reserve a separate margin for shared spectral colorbars.
    colorbars = [ax for ax in fig.axes if ax.get_label() == "<colorbar>"]
    if colorbars and len(fig.axes) == 5:
        fig.subplots_adjust(right=0.86)
        colorbars[0].set_position([0.90, 0.15, 0.02, 0.70])
    for extension in ("png", "pdf"):
        fig.savefig(OUTPUT / f"{name}.{extension}", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)


TaylorSwift 0.0.0.dev0; outputs: c:\Users\paulinkenbrandt\Documents\Github\TaylorSwift\examples\outputs\toa5


## Load and check units
Replace `DATA_PATH` with your own TOA5 file when ready. Required channels are
`Ux`, `Uy`, `Uz` in m/s, `T_SONIC` in °C, `CO2_density` in mg/m³,
and `H2O_density` in g/m³. Timestamps must represent a regular sample grid;
this example treats them as UTC. Match sampling frequency and site geometry to
your deployment. For other column names, use `process_file(..., column_map={"your_u": "Ux", ...})`.

In [2]:
DATA_PATH = ROOT / "examples/data/TOA5_ExampleSite_2024_07_15_1200.dat"
raw, metadata = tswift.read_toa5(DATA_PATH)
display(metadata)
display(raw.head())
config = tswift.SiteConfig(z_measurement=3.0, z_canopy=0.3,
                          sampling_freq=20.0, averaging_period=30.0)
channels = ["Ux", "Uy", "Uz", "T_SONIC", "CO2_density", "H2O_density"]
assert all(c in raw.columns for c in ["TIMESTAMP", *channels])
print(f"{len(raw):,} rows; expected samples/interval: {int(60 * config.averaging_period * config.sampling_freq):,}")
display(raw.select(channels).null_count())

{'file_type': 'TOA5',
 'station_id': 'ExampleSite',
 'logger_model': 'CR3000',
 'serial': '8675',
 'os_version': 'OS32',
 'program': 'EddyFlux.CR3',
 'units': {'TIMESTAMP': 'TS',
  'RECORD': '',
  'Ux': 'm/s',
  'Uy': 'm/s',
  'Uz': 'm/s',
  'T_SONIC': 'C',
  'CO2_density': 'mg/m^3',
  'H2O_density': 'g/m^3'}}

TIMESTAMP,RECORD,Ux,Uy,Uz,T_SONIC,CO2_density,H2O_density
datetime[μs],i64,f64,f64,f64,f64,f64,f64
2024-07-15 12:00:00,0,5.0708,0.1187,-0.0768,24.5931,753.0799,12.5645
2024-07-15 12:00:00.050,1,3.943,0.032,-0.1237,25.3253,753.7163,12.2716
2024-07-15 12:00:00.100,2,3.6672,0.2464,0.0175,25.1093,754.3571,12.1118
2024-07-15 12:00:00.150,3,4.3395,0.2135,0.0434,25.3156,742.7142,11.8402
2024-07-15 12:00:00.200,4,3.7648,0.0722,-0.0413,25.7475,745.6708,12.4841


36,000 rows; expected samples/interval: 36,000


Ux,Uy,Uz,T_SONIC,CO2_density,H2O_density
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


## Despike and inspect the changes
UKDE returns a cleaned copy. Inspect changes before using these settings on field
data: unusual turbulence is not necessarily an instrument spike. Raw-data screening
inside `process_file` records diagnostics; it does not perform this despiking step.

In [ ]:
# Start gently; adjust these before rerunning this cell.
DESPIKE_ENABLED = True
DESPIKE_COLUMNS = channels  # Use e.g. ["Uz"] to clean only one sensor.
PROB_THRESHOLD = 1e-6       # Lower = fewer removals.
MAX_ITER = 1               # More passes can progressively trim real tails.
BULK_IQR = 8.0             # Larger = broader fit; None fits all finite values.

clean = tswift.despike_dataframe(
    raw, columns=DESPIKE_COLUMNS, prob_threshold=PROB_THRESHOLD,
    max_iter=MAX_ITER if DESPIKE_ENABLED else 0,
    bulk_iqr=BULK_IQR, verbose=True,
)
seconds = np.arange(len(raw)) / config.sampling_freq
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for ax, channel, unit in zip(axes, ["Uz", "T_SONIC"], ["m/s", "°C"]):
    ax.plot(seconds, raw[channel], color="0.65", label="Raw", linewidth=0.7)
    ax.plot(seconds, clean[channel], color="tab:blue", label="Despiked", linewidth=0.6)
    ax.set_ylabel(f"{channel} [{unit}]")
    ax.legend()
axes[-1].set_xlabel("Elapsed time [s]")
fig.tight_layout()
save_plot(fig, "raw_and_despiked")

## Process and inspect quality diagnostics
`process_file` partitions the data, screens missing values, rotates wind, detrends,
computes windowed FFTs, and bins spectra. `run_qc` adds slope and low-friction-velocity
diagnostics in place; it does not discard results. Synthetic signals need not match
Kaimal slopes. These diagnostics alone are not a complete stationarity or field QC assessment.

In [ ]:
results = tswift.run_qc(tswift.process_file(clean, config))
if not results or not any(len(r.freq) for r in results):
    raise ValueError("No usable spectra: check sampling rate, coverage, units, and missing values.")
display(tswift.results_to_dataframe(results).select(
    "timestamp_start", "u_mean", "ustar", "zL", "H", "interval_status", "ustar_filter"))
display(pd.Series(results[0].qc_flags, name="First interval diagnostics"))

## Spectra, cospectra, and ogives
The horizontal coordinate is dimensionless frequency `n × (z − d) / U`.
Cospectra show shared variability between vertical wind and a scalar or longitudinal
wind. The helper plots absolute normalized cospectra on log axes, so read flux signs
from the table. Power spectra show variance by scale. An ogive approaching a plateau
at low frequency suggests convergence within the available averaging period.

Use an unrestricted stability range here so synthetic intervals are visible.
The Kaimal curves and reference slopes are comparison guides, not fitted results.

In [ ]:
for name, plotter in [("cospectra", tswift.plot_cospectra),
                      ("power_spectra", tswift.plot_spectra),
                      ("ogives", tswift.plot_ogive)]:
    fig, axes = plotter(results, stability_range=(-np.inf, np.inf))
    save_plot(fig, name)

## Apply spectral corrections and compare estimates
The instrument defaults represent an integrated IRGASON; change response times and
sensor geometry for your instrument. Correction factors use model cospectra.
Raw arrays and flux fields remain unchanged: corrected scalar values live in
`qc_flags`, and frequency-wise corrected arrays live in `corrected_spectra`.
These are different estimators and need not integrate to the same flux.

WPL is explicitly disabled because the sample has no measured pressure. For field
data with `PA` in kPa, first call `enrich_results_with_means(results, clean, config)`,
then enable `apply_wpl=True` and inspect `wpl_status` and `wpl_pressure_source`.

In [ ]:
instrument = tswift.InstrumentConfig(tau_co2=0.1, tau_h2o=0.1)
results = tswift.apply_spectral_corrections(
    results, config, instrument, apply_high_freq=True, apply_low_freq=True,
    apply_wpl=False, method="massman")
table = tswift.results_to_dataframe(results)
display(table.select("timestamp_start", "cf_wT", "cf_wCO2", "H", "H_corrected",
                     "spectral_status", "wpl_status"))
r = results[0]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(["Raw", "Model corrected"], [r.H, r.qc_flags["H_corrected"]])
axes[0].set_ylabel("Sensible heat flux [W/m²]")
axes[1].loglog(r.freq, np.abs(r.cosp_wCO2), label="Raw")
axes[1].loglog(r.freq, np.abs(r.corrected_spectra["cosp_wCO2"]), label="Deconvolved")
axes[1].set_xlabel("Frequency [Hz]")
axes[1].set_ylabel("|n Co(w, CO₂)| [mg m⁻² s⁻¹]")
axes[1].legend()
fig.tight_layout()
save_plot(fig, "correction_comparison")

## Export results
The interval table includes diagnostics and corrected scalar estimates. The long
spectral table exports **raw** arrays. Export the corrected CO₂ arrays explicitly
below to keep their meaning clear. CSV is convenient for inspection; Parquet retains types.

In [ ]:
tswift.results_to_csv(results, OUTPUT / "intervals.csv")
tswift.results_to_parquet(results, OUTPUT / "intervals.parquet")
tswift.spectra_to_dataframe(results).write_parquet(OUTPUT / "raw_spectra.parquet")
pl.DataFrame({"freq_hz": r.freq,
              "raw_nCo_wCO2": r.cosp_wCO2,
              "deconvolved_nCo_wCO2": r.corrected_spectra["cosp_wCO2"]
             }).write_csv(OUTPUT / "first_interval_co2_spectra.csv")
display(sorted(p.name for p in OUTPUT.iterdir()))